# V2 Monte Carlo
### *Model-free RL, on-policy method*

#### Load the Tic-Tac-Toe environment.

In [1]:
from tic_tac_toe_env import TicTacToe
import copy
import numpy as np
import random

#### Declare a random policy and the rule-based policy from Lecture for benchmarking. 


In [2]:
def random_move(game: TicTacToe, letter: str):
    board = game.board
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    move = random.choice(game.available_moves())
    return move

In [3]:
def choose_move(game: TicTacToe, letter: str):
    # board = list(game.get_flat_state())
    board = game.board
    n = game.n
    player = letter
    opponent = 'O' if player == 'X' else 'X'

    def empty():
        return game.available_moves()

    def lines():
        all_lines = []
        for i in range(n):
            all_lines.append([i*n+j for j in range(n)])
        for j in range(n):
            all_lines.append([i*n+j for i in range(n)])
        # diag
        all_lines.append([i*n+i for i in range(n)])
        # off-diag
        all_lines.append([i*n+(n-1-i) for i in range(n)])
        return all_lines

    def can_win(marker):
        winning_moves = []
        for line in lines():
            vals = [board[i] for i in line]
            if vals.count(marker) == n-1 and vals.count('_') == 1:
                winning_moves.append(line[vals.index('_')])
        return winning_moves # return the list of indicies for winning moves

    # if our wins is not empty then play any of those moves to win (here just pick the first)
    our_wins = can_win(player)
    if our_wins:
        return our_wins[0]

    # if the opponent can win then we just block the move. From the lecture we should just pick it randomly
    opp_wins = can_win(opponent)
    if opp_wins:
        return np.random.choice(opp_wins).item()
    
    empty_cells = empty()

    first_empty = empty_cells[0]
    
    # play sequentially in the row first
    current_row = first_empty//n # recall that these are flattened indices
    next_in_row = first_empty + 1 # next cell in the same row
    
    # if its truly on the same row and empty then play it, otherwise it might wrap around and not make snese
    if next_in_row < (current_row + 1) * n and next_in_row in empty_cells:
        return next_in_row
    
    # now, if thats the case, then try the cell below
    cell_below = first_empty + n
    if cell_below < n*n and cell_below in empty_cells: # so if its actually valid (which it should be) and its empty then play it
        return cell_below
    
    # else play randomly
    return np.random.choice(empty_cells).item()

#### Setup the Monte Carlo Agent.

In [ ]:
import copy
import random

class MonteCarloAgent:
    def __init__(self, letter, episodes=20000, alpha=0.1, gamma=1.0, epsilon=0.1):

        self.letter = letter
        self.opponent = 'O' if letter == 'X' else 'X'
        self.episodes = episodes
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon 
        self.q = {}  # storing state-action values
        self.wins = []
        self.losses = []
        self.draws  = []

    def get_move(self, game):
        """
        Choose the best move based on learned Q-values, break ties randomly.
        """
        moves = game.available_moves()
        if not moves:
            return None

        state = self._serialize_board(game)
        q_values = [self.q.get((state, a), 0.1) for a in moves]  
        max_q = max(q_values)

        # Pick randomly from the best moves
        best_moves = [m for m, qv in zip(moves, q_values) if qv == max_q]
        return random.choice(best_moves)

    def train(self):
        """
        Run episodes using epsilon-greedy exploration and Monte Carlo updates.
        """
        self.win_count = 0
        
        self.loss_count = 0
        
        self.draw_count = 0
        
        for _ in range(self.episodes):
            game = self._init_game()
            trajectory = self._generate_episode(game)

            # Determine final outcome for statistics
            if game.current_winner == self.letter:
                self.win_count += 1
            elif game.current_winner == self.opponent:
                self.loss_count += 1
            else:
                self.draw_count += 1
            # compute returns and update Q-values incrementally
            G = 0
            for state, action, reward in reversed(trajectory):
                G = reward + self.gamma * G
                key = (state, action)
                if key not in self.q:
                    self.q[key] = 0.1  # small positive init
                self.q[key] += self.alpha * (G - self.q[key])

            if (_+1) % 10000 == 0:
                total = self.win_count + self.loss_count + self.draw_count
                self.wins.append(self.win_count/total)
                self.losses.append(self.loss_count/total)
                self.draws.append(self.draw_count/total)
                print(f"After {_+1} episodes: "
                f"W {self.win_count/total:.2%}, "
                f"L {self.loss_count/total:.2%}, "
                f"D {self.draw_count/total:.2%}")
        
        # Print summary stats
        print("Training complete.")
        print(f"Wins:  {self.win_count} ({self.win_count/self.episodes:.2%})")
        print(f"Losses:{self.loss_count} ({self.loss_count/self.episodes:.2%})")
        print(f"Draws: {self.draw_count} ({self.draw_count/self.episodes:.2%})")



    def _generate_episode(self, game):
        """
        Play a full episode using epsilon-greedy policy for agent and random for opponent.
        """
        trajectory = []
        turn = self.letter  # agent alwaysstarts for consistency

        while game.empty_squares() and not game.current_winner:
            state = self._serialize_board(game)
            moves = game.available_moves()

            if turn == self.letter:
                # epsilon-greedy: explore with probability epsilon
                if random.random() < self.epsilon:
                    move = random.choice(moves)
                else:
                    # pick move with max Q-value (default 0.1)
                    q_values = [self.q.get((state, a), 0.1) for a in moves]
                    max_q = max(q_values)
                    best_moves = [m for m, qv in zip(moves, q_values) if qv == max_q]
                    move = random.choice(best_moves)
            else:
                # opponent moves randomly
                move = random.choice(moves)

            game.make_move(move, turn)

            # reward 0 for intermediate moves; actual reward will be calculated at end
            trajectory.append((state, move, 0))
            turn = self.letter if turn == self.opponent else self.opponent

        # Assign step-wise rewards for final outcome
        final_reward = 0
        if game.current_winner == self.letter:
            final_reward = 1
        elif game.current_winner == self.opponent:
            final_reward = -1

        # propagate final reward to all steps
        trajectory = [(s, a, final_reward) for s, a, _ in trajectory]
        return trajectory

    def _serialize_board(self, game):
        return tuple(cell for row in game.board for cell in row)

    def _init_game(self):
        return copy.deepcopy(self.game_template)

    def set_game_template(self, game):
        self.game_template = copy.deepcopy(game)

    def save_q(self, filename="q_values.pkl"):
        import pickle
        with open(filename, "wb") as f:
            pickle.dump(self.q, f)
        print(f"Q-values saved to '{filename}'")

    def load_q(self, filename="q_values.pkl"):
        import pickle
        with open(filename, "rb") as f:
            self.q = pickle.load(f)
        print(f"Q-values loaded from '{filename}'")

#### Train the Monte Carlo Agent

In [136]:
board_size = 4
num_episodes = 200000
# so ai learns the optimal policy when it plays as X
ai = MonteCarloAgent('X', episodes=num_episodes, alpha=0.1, gamma=1.0, epsilon=0.1)

game = TicTacToe(n=board_size)  # create a new Tic-Tac-Toe board
ai.set_game_template(game)

ai.train()
print("Number of q-values learned:", len(ai.q))
print(ai.get_move(game))
# ai.save_q(f"tictactoe_mc_{board_size}x{board_size}_eps={num_episodes}.pkl")

After 10000 episodes: W 34.58%, L 25.34%, D 40.08%
After 20000 episodes: W 34.72%, L 24.89%, D 40.39%
After 30000 episodes: W 35.08%, L 24.89%, D 40.03%
After 40000 episodes: W 35.45%, L 24.77%, D 39.78%
After 50000 episodes: W 35.81%, L 24.56%, D 39.63%
After 60000 episodes: W 36.25%, L 24.35%, D 39.40%
After 70000 episodes: W 36.39%, L 24.21%, D 39.40%
After 80000 episodes: W 36.64%, L 24.11%, D 39.25%
After 90000 episodes: W 36.95%, L 23.89%, D 39.16%
After 100000 episodes: W 37.22%, L 23.71%, D 39.07%
After 110000 episodes: W 37.50%, L 23.51%, D 39.00%
After 120000 episodes: W 37.76%, L 23.40%, D 38.84%
After 130000 episodes: W 38.07%, L 23.30%, D 38.63%
After 140000 episodes: W 38.30%, L 23.15%, D 38.55%
After 150000 episodes: W 38.59%, L 23.01%, D 38.40%
After 160000 episodes: W 38.89%, L 22.87%, D 38.24%
After 170000 episodes: W 39.16%, L 22.75%, D 38.09%
After 180000 episodes: W 39.42%, L 22.64%, D 37.94%
After 190000 episodes: W 39.58%, L 22.57%, D 37.85%
After 200000 episodes

In [137]:
ai.save_q("dec6_mc_4x4")
# print(ai.wins)

Q-values saved to 'dec6_mc_4x4'


#### Evaluation

In [ ]:
def play_opp_first(episode, n):

    game = TicTacToe(n=n)
    # assign player letters
    human_letter = 'O'
    ai_letter = 'X'

    while game.empty_squares():
         # TURN 1: OPP
        move = None
        while move not in game.available_moves():
          move = random_move(game, human_letter)
        game.make_move(move, human_letter)

        if game.current_winner:
            return -1

        if not game.empty_squares():
            return 0
 
        # TURN 2: AI
        ai_move = ai.get_move(game)

        game.make_move(ai_move, ai_letter)

        if game.current_winner:
            return 1

        if not game.empty_squares():
            return 0

In [ ]:
def play_ai_first(episode, n):

    game = TicTacToe(n=n)
    # assign player letters
    human_letter = 'O'
    ai_letter = 'X'

    while game.empty_squares():

        # TURN 1: AI
        ai_move = ai.get_move(game)

        game.make_move(ai_move, ai_letter)

        if game.current_winner:
            return 1

        if not game.empty_squares():
            return 0
        
         # TURN 2: OPP
        move = None
        while move not in game.available_moves():
          move = random_move(game, human_letter)
        game.make_move(move, human_letter)

        if game.current_winner:
            return -1

        if not game.empty_squares():
            return 0
 


## Evaluating.

In [ ]:
board_size = 8
numAIWins = 0
numOppWins = 0
ai_wins = []
ai_losses = []
ai_draws = []
print("Board size: ", board_size)

for i in range(15000): 
  # returns 0 for tie, 1 for AI win, -1 for AI lose
  result_opp_first = play(i, board_size)
  result_ai_first = play_ai_first(i, board_size)
  if result_opp_first == 1:
    numAIWins += 1
    # print("AI Won")
  if result_opp_first == -1:
    numOppWins += 1
  if result_ai_first == 1:
    numAIWins += 1
    # print("AI Won")
  if result_ai_first == -1:
    numOppWins += 1
    # print("Opp won")

  if (i+1) % 1000 == 0:
    # print("----------------", i, "----------------")
    # print("Number of AI Wins: ", numAIWins)
    # print("Number of Draws: ", 15000-numAIWins-numOppWins)
    # print("Number of Opp Wins: ", numOppWins)
    ai_wins.append(numAIWins/30000)
    ai_losses.append(numOppWins/30000)
    ai_draws.append((30000-numAIWins-numOppWins)/30000) 

print("Number of AI Wins: ", numAIWins)
print("Number of Draws: ", 15000-numAIWins-numOppWins)
print("Number of Opp Wins: ", numOppWins)
print("Wins:", ai_wins)
print("Losses:", ai_losses)
print("Draws:", ai_draws)

# print("Win Rate: ", numAIWins/1500)
# print("Draw Rate: ", (1500-numAIWins-numOppWins)/1500)
# print("Opp Wins: ", numOppWins/1500)


Board size:  7
Number of AI Wins:  2175
Number of Draws:  10669
Number of Opp Wins:  2156
Wins: [0.0055, 0.0099, 0.014166666666666666, 0.0191, 0.024033333333333334, 0.029133333333333334, 0.034166666666666665, 0.03883333333333333, 0.0433, 0.04816666666666667, 0.0531, 0.058133333333333335, 0.0627, 0.06756666666666666, 0.0725]
Losses: [0.0061, 0.011166666666666667, 0.0158, 0.020966666666666668, 0.0256, 0.030033333333333332, 0.0351, 0.04016666666666667, 0.044333333333333336, 0.0493, 0.0541, 0.05823333333333333, 0.0627, 0.06706666666666666, 0.07186666666666666]
Draws: [0.9884, 0.9789333333333333, 0.9700333333333333, 0.9599333333333333, 0.9503666666666667, 0.9408333333333333, 0.9307333333333333, 0.921, 0.9123666666666667, 0.9025333333333333, 0.8928, 0.8836333333333334, 0.8746, 0.8653666666666666, 0.8556333333333334]


In [131]:
# Load trained Q-values
ai.load_q("tictactoe_q_5x5.pkl")

numAIWins = 0
numHumanWins = 0
numTies = 0

for i in range(15000):
    result = play(ai, human_policy=random_human_policy)
    if result == 1:
        numAIWins += 1
    elif result == -1:
        numHumanWins += 1
    else:
        numTies += 1

print(f"AI wins: {numAIWins}")
print(f"Human wins: {numHumanWins}")
print(f"Ties: {numTies}")


Q-values loaded from 'tictactoe_q_5x5.pkl'
AI wins: 3878
Human wins: 2315
Ties: 8807


## Checking play against human.

In [139]:
def play():
    print("Welcome to Tic Tac Toe! You are O. AI is X.")

    # declare the tic tac toe environment
    game = TicTacToe(n=4)
    
    # assign player letters
    human_letter = 'O'
    ai_letter = 'X'

    game.print_board()

    while game.empty_squares():
        # TURN 1: HUMAN
        move = None
        while move not in game.available_moves():
            try:
                move = int(input("Enter your move (0-b): "))
            except ValueError:
                continue
        game.make_move(move, human_letter)
        game.print_board()

        if game.current_winner:
            print("You win!")
            return

        if not game.empty_squares():
            print("It's a tie!")
            return

        # TURN 2: AI
        # the get_move function is where we run the monte carlo simulation
        ai_move = ai.get_move(game)

        game.make_move(ai_move, ai_letter)
        print(f"AI moves at {ai_move}")
        game.print_board()

        if game.current_winner:
            print("AI wins!")
            return

        if not game.empty_squares():
            print("It's a tie!")
            return

In [138]:
ai.load_q("dec6_mc_4x4")

Q-values loaded from 'dec6_mc_4x4'


In [140]:
play()

Welcome to Tic Tac Toe! You are O. AI is X.
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
| O |   |   |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   |   |
AI moves at 15
| O |   |   |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   | X |
| O |   | O |   |
|   |   |   |   |
|   |   |   |   |
|   |   |   | X |
AI moves at 11
| O |   | O |   |
|   |   |   |   |
|   |   |   | X |
|   |   |   | X |
| O | O | O |   |
|   |   |   |   |
|   |   |   | X |
|   |   |   | X |
AI moves at 4
| O | O | O |   |
| X |   |   |   |
|   |   |   | X |
|   |   |   | X |
| O | O | O | O |
| X |   |   |   |
|   |   |   | X |
|   |   |   | X |
You win!
